# Перенос PhysTwin в Newton своими руками

Практикум по реализации и проверке адаптера **mass–spring → SolverSemiImplicit**.
Язык объяснений — русский, код — Python. Предполагается знакомство с NumPy,
векторами и вторым законом Ньютона; знание Warp заранее не требуется.

**Результат:** ты соберёшь граф частиц и пружин в Newton, проверишь силы и шаг
интегрирования, воспроизведёшь управление, drag и столкновение с плоскостью,
проверишь градиент и подготовишь перенос своего обученного объекта.

Это учебная проверка реализации на синтетических примерах. Она не измеряет
качество калибровки по RGB-D или sim-to-real. Полное соответствие PhysTwin
потребует отдельной проверки self-collision и реальных записей.

## Как проходить

1. Выбери ядро **Python (PhysTwin → Newton)** на хосте по разделу **«Notebook: перенос PhysTwin»**
   в [environments/README.md](../environments/README.md). Сам notebook лежит в `tutorials/`.
2. Выполняй ячейки сверху вниз. В заданиях заменяй `raise NotImplementedError` своим кодом.
3. Запускай проверку после задания. `TODO` означает незавершённое задание, **не успешный тест**.
   Ошибки реализации не скрываются. После исправления повтори зависимые ячейки.
4. Решения спрятаны под раскрывающимися подсказками. Они не исполняются автоматически.
5. `Restart Kernel → Run All` проверяет, что результат не зависит от старых переменных.

Основная часть работает на CPU; CUDA можно выбрать явно. Встроенные графики и ползунок
не требуют X11. Данные, кэши и необязательные экспорты хранятся вне репозитория.

## Источники и границы версии

- [PhysTwin: исходный симулятор](https://github.com/Jianghanxiao/PhysTwin/blob/2b6630528141b9cba5a7677c8b88b2129b4a8390/qqtt/model/diff_simulator/spring_mass_warp.py).
- [PhysTwin: восстановление checkpoint](https://github.com/Jianghanxiao/PhysTwin/blob/2b6630528141b9cba5a7677c8b88b2129b4a8390/qqtt/engine/trainer_warp.py).
- [Newton 1.6.1: решатели и дифференцирование](https://newton-physics.github.io/newton/1.6.1/solvers/index.html).
- [Существующий независимый перенос](https://github.com/BaochaiXue/Newton/blob/dff77ddfc4ec014750ae3f98d54db64a10137683/phystwin_bridge/tools/core/newton_import_ir.py)
  — материал для сравнения после собственной реализации.

Урок рассчитан на `newton==1.6.1`, `warp-lang==1.17.0` в отдельном venv хоста.
Здесь используется Newton напрямую; подключение адаптера к среде Isaac Lab — следующий этап.

In [7]:
from pathlib import Path
from importlib import metadata
import os
import json
import hashlib
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import warp as wp
import newton
from newton.solvers import SolverSemiImplicit

versions = {name: metadata.version(name) for name in ('newton', 'warp-lang', 'numpy')}
print(versions)
assert versions['newton'] == '1.6.1', 'Выбери ядро Python (PhysTwin → Newton) из environments/README.md'
assert versions['warp-lang'] == '1.17.0', 'Урок проверен с Warp 1.17.0'
DEVICE = os.environ.get('LESSON_DEVICE', 'cuda:0')  # или явно 'cuda:0'
CACHE = Path(os.environ.get('LESSON_CACHE', Path.home() / '.cache/deformable-twins/phystwin-lesson'))
CACHE.mkdir(parents=True, exist_ok=True)
wp.config.kernel_cache_dir = str(CACHE / 'warp')
wp.init()
print('Device:', wp.get_device(DEVICE))

passed = set()
def check(name, test):
    passed.discard(name)
    try:
        test()
    except NotImplementedError as error:
        print(f'TODO {name}: {error}')
        return
    passed.add(name)
    print(f'PASS {name}')

def ready(*names):
    missing = set(names) - passed
    if missing:
        print('Сначала выполни задания и проверки:', ', '.join(sorted(missing)))
    return not missing

{'newton': '1.6.1', 'warp-lang': '1.17.0', 'numpy': '2.2.6'}
Device: cuda:0


## 1. Сначала сопоставь уравнения

Пусть $d=(x_j-x_i)/\|x_j-x_i\|$, $\ell=\|x_j-x_i\|$, $\ell_0$ — длина покоя.
На частицу $i$ PhysTwin действует сила:

$$F_i=Y(\ell/\ell_0-1)d+c\big((v_j-v_i)\cdot d\big)d.$$

Newton использует $k(\ell-\ell_0)d$ и такой же продольный dashpot.
Отсюда **$k=Y/\ell_0$**, а $c$ переносится без деления на длину.
Название `spring_Y` не делает параметр модулем Юнга в Па:
в этой формуле при координатах в метрах $Y$ имеет размерность силы, а $k$ — Н/м.

В runtime PhysTwin хранится **log(Y)**. В checkpoint `spring_Y` уже хранится
линейный $Y$ — нельзя делать `exp` повторно. В исходном kernel пружина активна,
только если `exp(log_Y) > Y_min`; затем значение ограничивается сверху/снизу.
У отключённой пружины отсутствуют **и упругая сила, и dashpot**.

### Задание 1 — коэффициенты

Верни три массива: `ke`, `kd`, `active`. Учти выключение пружин и верхнюю границу.
Проверь, что длины покоя положительны. Контрольный пример:
`Y=4 N`, `rest=0.2 m` → `ke=20 N/m`.

In [8]:
def spring_coefficients(log_y, rest, damping, y_min, y_max):
    log_y = np.asarray(log_y, dtype=np.float64)
    rest = np.asarray(rest, dtype=np.float64)
    if log_y.shape != rest.shape or np.any(rest <= 0):
        raise ValueError('Несовместимые формы или неположительные rest lengths')
    raw_y = np.exp(log_y)
    if not np.isfinite(raw_y).all():
        raise ValueError('Нефинитное Y')
    active = raw_y > y_min
    ke = np.where(active, np.clip(raw_y, y_min, y_max) / rest, 0.0)
    kd = np.where(active, damping, 0.0)
    return ke, kd, active

<details><summary>Подсказка: раскрыть рабочий вариант после своей попытки</summary>

Скопируй код в ячейку задания и выполни её повторно.

```python
def spring_coefficients(log_y, rest, damping, y_min, y_max):
    log_y = np.asarray(log_y, dtype=np.float64)
    rest = np.asarray(rest, dtype=np.float64)
    if log_y.shape != rest.shape or np.any(rest <= 0):
        raise ValueError('Несовместимые формы или неположительные rest lengths')
    raw_y = np.exp(log_y)
    if not np.isfinite(raw_y).all():
        raise ValueError('Нефинитное Y')
    active = raw_y > y_min
    ke = np.where(active, np.clip(raw_y, y_min, y_max) / rest, 0.0)
    kd = np.where(active, damping, 0.0)
    return ke, kd, active
```

</details>

In [9]:
def test_coefficients():
    ke, kd, active = spring_coefficients(np.log([0.5, 1., 4., 100.]),
                                         np.array([.2, .2, .2, .5]), .3, 1., 10.)
    np.testing.assert_allclose(ke, [0., 0., 20., 20.])
    np.testing.assert_allclose(kd, [0., 0., .3, .3])
    np.testing.assert_array_equal(active, [False, False, True, True])
    try:
        spring_coefficients(np.log([4.]), np.array([0.]), .3, 1., 10.)
    except ValueError:
        pass
    else:
        raise AssertionError('Нулевая длина покоя должна быть отклонена')
check('coefficients', test_coefficients)

PASS coefficients


## 2. Учебный контракт данных

Первые `n_object` частиц — динамический объект, оставшиеся — кинематические
контроллеры. Их массу в Newton задаём равной нулю. Порядок вершин и рёбер
сохраняем; создание нового графа соседства изменило бы модель.

`q`, `v`, `rest`, `mass` используют метры, секунды и килограммы.
В этих тестах Z направлена вверх. Для своего PhysTwin экспортируй фактическую
гравитацию с учётом `reverse_factor`; не переворачивай координаты без преобразования
траекторий и камер.

Следующий код — готовая инфраструктура урока. NumPy-эталон независимо вычисляет
формулу PhysTwin и нужен для проверки Newton. Это не запуск полного PhysTwin.

In [10]:
def fixture():
    return dict(
        q=np.array([[0., 0., .8], [.13, 0., .8], [.23, 0., .8]], dtype=np.float64),
        v=np.array([[.02, 0., 0.], [-.03, 0., 0.], [0., 0., 0.]]),
        mass=np.array([.1, .15, 0.]), n_object=2,
        edges=np.array([[0, 1], [1, 2]], dtype=np.int32),
        rest=np.array([.1, .1]), log_y=np.log([3., 4.]),
        damping=.15, y_min=1., y_max=100., drag=0.,
        gravity=np.array([0., 0., -9.8]), dt=1e-4,
    )

def validate_graph(g):
    q, v, mass = g['q'], g['v'], g['mass']
    n, e = len(q), len(g['edges'])
    if q.shape != (n, 3) or v.shape != q.shape or mass.shape != (n,):
        raise ValueError('Неверная форма q / v / mass')
    if not 0 < g['n_object'] <= n:
        raise ValueError('Неверное число object particles')
    if np.any(mass[:g['n_object']] <= 0) or np.any(mass[g['n_object']:] != 0):
        raise ValueError('Объекту нужны положительные массы, контроллерам — нулевые')
    edges = g['edges']
    if edges.shape != (e, 2) or not np.issubdtype(edges.dtype, np.integer):
        raise ValueError('edges должен быть целочисленным массивом (E, 2)')
    if np.any(edges < 0) or np.any(edges >= n) or np.any(edges[:, 0] == edges[:, 1]):
        raise ValueError('Неверные индексы рёбер')
    if g['rest'].shape != (e,) or g['log_y'].shape != (e,) or np.any(g['rest'] <= 0):
        raise ValueError('Неверные rest / log_y')
    for key in ('q', 'v', 'mass', 'rest', 'log_y', 'gravity', 'dt', 'damping', 'drag', 'y_min', 'y_max'):
        if not np.isfinite(g[key]).all():
            raise ValueError(f'Нефинитные данные: {key}')
    if g['gravity'].shape != (3,) or g['dt'] <= 0 or min(g['damping'], g['drag'], g['y_min']) < 0:
        raise ValueError('Неверные параметры интегрирования')
    if g['y_max'] <= g['y_min']:
        raise ValueError('Неверные границы Y')
    if e and np.any(np.linalg.norm(q[edges[:, 1]] - q[edges[:, 0]], axis=1) <= 1e-6):
        raise ValueError('Вырожденная пружина: Newton 1.6.1 делит на её текущую длину')

def phystwin_forces_numpy(g, q, v):
    f = np.zeros_like(q, dtype=np.float64)
    for edge, rest, log_y in zip(g['edges'], g['rest'], g['log_y']):
        raw_y = np.exp(log_y)
        if raw_y <= g['y_min']:
            continue
        i, j = edge
        delta = q[j] - q[i]
        length = np.linalg.norm(delta)
        direction = delta / max(length, 1e-6)
        force = (np.clip(raw_y, g['y_min'], g['y_max']) * (length / rest - 1.)
                 + g['damping'] * np.dot(v[j] - v[i], direction)) * direction
        if i < g['n_object']:
            f[i] += force
        if j < g['n_object']:
            f[j] -= force
    return f

g = fixture()
validate_graph(g)
print('Эталонные силы на объект:', phystwin_forces_numpy(g, g['q'], g['v'])[:g['n_object']])

Эталонные силы на объект: [[ 0.8925  0.      0.    ]
 [-0.888   0.      0.    ]]


### Задание 2 — ModelBuilder

Создай `ModelBuilder(gravity=...)`, добавь частицы через `add_particle` и активные
пружины через `add_spring(i, j, ke, kd, control=0.0)`. Затем вызови
`finalize(device=DEVICE, requires_grad=requires_grad)`.

**Ловушка:** `add_spring` вычисляет rest length по текущим координатам.
Перезапиши `builder.spring_rest_length[-1]` экспортированным значением.
Отключи particle contacts через `model.particle_grid = None`. Плоскостей и других
геометрий пока нет. Увеличь `model.particle_max_velocity` до `1e6`, чтобы штатный
ограничитель скорости не менял учебное сравнение; расходимость проверим отдельно.

In [11]:
def build_model(g, requires_grad=False):
    validate_graph(g)
    builder = newton.ModelBuilder(gravity=tuple(g['gravity']))
    for q, v, mass in zip(g['q'], g['v'], g['mass']):
        builder.add_particle(pos=tuple(q), vel=tuple(v), mass=float(mass), radius=.001)
    ke, kd, active = spring_coefficients(g['log_y'], g['rest'], g['damping'], g['y_min'], g['y_max'])
    for index in np.flatnonzero(active):
        i, j = g['edges'][index]
        builder.add_spring(int(i), int(j), float(ke[index]), float(kd[index]), control=0.0)
        builder.spring_rest_length[-1] = float(g['rest'][index])
    model = builder.finalize(device=DEVICE, requires_grad=requires_grad)
    model.particle_grid = None
    model.particle_max_velocity = 1e6
    return model

<details><summary>Подсказка: раскрыть рабочий вариант после своей попытки</summary>

Скопируй код в ячейку задания и выполни её повторно.

```python
def build_model(g, requires_grad=False):
    validate_graph(g)
    builder = newton.ModelBuilder(gravity=tuple(g['gravity']))
    for q, v, mass in zip(g['q'], g['v'], g['mass']):
        builder.add_particle(pos=tuple(q), vel=tuple(v), mass=float(mass), radius=.001)
    ke, kd, active = spring_coefficients(g['log_y'], g['rest'], g['damping'], g['y_min'], g['y_max'])
    for index in np.flatnonzero(active):
        i, j = g['edges'][index]
        builder.add_spring(int(i), int(j), float(ke[index]), float(kd[index]), control=0.0)
        builder.spring_rest_length[-1] = float(g['rest'][index])
    model = builder.finalize(device=DEVICE, requires_grad=requires_grad)
    model.particle_grid = None
    model.particle_max_velocity = 1e6
    return model
```

</details>

In [12]:
def test_builder():
    g = fixture()
    model = build_model(g)
    assert model.particle_count == 3 and model.spring_count == 2
    np.testing.assert_allclose(model.spring_rest_length.numpy(), [.1, .1])
    np.testing.assert_allclose(model.spring_stiffness.numpy(), [30., 40.])
    np.testing.assert_allclose(model.particle_inv_mass.numpy(), [10., 1/.15, 0.], rtol=1e-6)
    assert model.particle_grid is None
    g['log_y'][0] = np.log(.5)
    assert build_model(g).spring_count == 1, 'Отключённая пружина должна исчезнуть'
if ready('coefficients'):
    check('builder', test_builder)

Module newton._src.geometry.bvh 58a3a49 load on device 'cuda:0' took 376.66 ms  (compiled)
PASS builder


## 3. Один шаг важнее красивого видео

Без drag и контактов оба вычисления должны дать:

$$v_{t+1}=v_t+\Delta t(F_t/m+g),\qquad x_{t+1}=x_t+\Delta t\,v_{t+1}.$$

### Задание 3 — шаг Newton

Обнули силы `state_in.clear_forces()` и вызови `solver.step(...)` с
`contacts=None`. Состояния входа и выхода должны быть разными объектами.
Не добавляй силы пружин вручную: SemiImplicit уже вычисляет их.

In [13]:
def native_step(solver, state_in, state_out, control, dt):
    if state_in is state_out:
        raise ValueError('Нужны разные входное и выходное состояния')
    state_in.clear_forces()
    solver.step(state_in, state_out, control, None, dt)

<details><summary>Подсказка: раскрыть рабочий вариант после своей попытки</summary>

Скопируй код в ячейку задания и выполни её повторно.

```python
def native_step(solver, state_in, state_out, control, dt):
    if state_in is state_out:
        raise ValueError('Нужны разные входное и выходное состояния')
    state_in.clear_forces()
    solver.step(state_in, state_out, control, None, dt)
```

</details>

In [14]:
def test_step():
    g = fixture()
    model = build_model(g)
    solver = SolverSemiImplicit(model, enable_tri_contact=False)
    a, b = model.state(), model.state()
    a.particle_f.fill_(wp.vec3(100., 100., 100.))  # проверяем clear_forces
    native_step(solver, a, b, model.control(), g['dt'])
    n = g['n_object']
    f = phystwin_forces_numpy(g, g['q'], g['v'])[:n]
    v_expected = g['v'][:n] + g['dt'] * (f / g['mass'][:n, None] + g['gravity'])
    q_expected = g['q'][:n] + g['dt'] * v_expected
    np.testing.assert_allclose(a.particle_f.numpy()[:n], f, atol=2e-5)
    np.testing.assert_allclose(b.particle_qd.numpy()[:n], v_expected, atol=2e-6)
    np.testing.assert_allclose(b.particle_q.numpy()[:n], q_expected, atol=2e-7)
    np.testing.assert_allclose(b.particle_q.numpy()[n:], g['q'][n:], atol=1e-7)
if ready('builder'):
    check('step', test_step)

Module newton._src.solvers.semi_implicit.kernels_particle 20846f5 load on device 'cuda:0' took 164.04 ms  (compiled)
Module newton._src.solvers.solver d7b7ff4 load on device 'cuda:0' took 96.88 ms  (compiled)
PASS step


## 4. Контроллеры PhysTwin

PhysTwin интерполирует заданные точки внутри видеокадра:
`alpha = (substep + 1) / num_substeps`. Контроллеры воздействуют на объект через
пружины; их координаты не следует навязывать частицам самого объекта.

В проверенной исходной реализации `wp_control_v` инициализируется нулями и
`set_controller_target` её не обновляет. Для совпадения перенеси **это поведение**.
Вычисление скорости как разности соседних кадров было бы отдельным изменением модели.

### Задание 4 — интерполяция управления

Верни интерполированные позиции и нулевые скорости контроллеров.
`substep` нумеруется с нуля; недопустимые индексы нужно отклонить.

In [15]:
def sample_controller(start, end, substep, num_substeps):
    if num_substeps <= 0 or not 0 <= substep < num_substeps:
        raise ValueError('Неверный substep')
    alpha = (substep + 1) / num_substeps
    return start + alpha * (end - start), np.zeros_like(start)

<details><summary>Подсказка: раскрыть рабочий вариант после своей попытки</summary>

Скопируй код в ячейку задания и выполни её повторно.

```python
def sample_controller(start, end, substep, num_substeps):
    if num_substeps <= 0 or not 0 <= substep < num_substeps:
        raise ValueError('Неверный substep')
    alpha = (substep + 1) / num_substeps
    return start + alpha * (end - start), np.zeros_like(start)
```

</details>

In [16]:
def test_controllers():
    start = np.array([[0., 0., 0.]])
    end = np.array([[1., 2., 3.]])
    q, v = sample_controller(start, end, 0, 4)
    np.testing.assert_allclose(q, end / 4)
    np.testing.assert_array_equal(v, np.zeros_like(start))
    np.testing.assert_allclose(sample_controller(start, end, 3, 4)[0], end)
    try:
        sample_controller(start, end, 4, 4)
    except ValueError:
        pass
    else:
        raise AssertionError('Индекс за пределами кадра должен быть отклонён')
check('controllers', test_controllers)

PASS controllers


## 5. Сопротивление: порядок операций

PhysTwin применяет $v'=(v+\Delta t(F/m+g))\exp(-\Delta t\,drag)$ **до** обновления
позиции и контакта с землёй. `angular_damping` у Newton относится к вращению
жёстких тел и не заменяет этот параметр.

### Задание 5 — поправка на drag

У тебя есть старая позиция и скорость **после** native Newton step.
Умножь скорости динамических частиц на экспоненту и заново вычисли позиции
из старых координат. Контроллеры не меняй.

Здесь используем NumPy, чтобы видеть порядок операций. Такой forward-адаптер
разрывает граф autodiff. Для градиентов через drag эту операцию нужно затем
переписать как Warp kernel с отдельными выходными буферами.

In [17]:
def apply_drag(q_old, native_v, n_object, drag, dt):
    q, v = q_old.copy(), native_v.copy()
    v[:n_object] *= np.exp(-dt * drag)
    q[:n_object] += dt * v[:n_object]
    return q, v

<details><summary>Подсказка: раскрыть рабочий вариант после своей попытки</summary>

Скопируй код в ячейку задания и выполни её повторно.

```python
def apply_drag(q_old, native_v, n_object, drag, dt):
    q, v = q_old.copy(), native_v.copy()
    v[:n_object] *= np.exp(-dt * drag)
    q[:n_object] += dt * v[:n_object]
    return q, v
```

</details>

In [18]:
def test_drag():
    q = np.zeros((2, 3)); v = np.ones((2, 3))
    q_out, v_out = apply_drag(q, v, 1, np.log(2.), 1.)
    np.testing.assert_allclose(v_out[0], .5)
    np.testing.assert_allclose(q_out[0], .5)
    np.testing.assert_array_equal(q_out[1], q[1])
    np.testing.assert_array_equal(v_out[1], v[1])
    np.testing.assert_array_equal(v, np.ones((2, 3)))
check('drag', test_drag)

PASS drag


## 6. Траектория и интерактивный просмотр

Этот готовый цикл связывает твои функции. На каждом подшаге он задаёт контроллеры,
считает Newton step, применяет drag и сохраняет кадры. Для контроля независимо
интегрируется NumPy-версия формул PhysTwin. Контакты пока отключены.

**Критерий учебной проверки задан заранее:** на этой небольшой сцене максимальное
расхождение координат за 400 подшагов должно быть меньше `2e-5 m`.
Это тест реализации, а не универсальная точность симулятора.

In [19]:
def rollout(g, controls, substeps=20):
    model = build_model(g)
    solver = SolverSemiImplicit(model, enable_tri_contact=False)
    a, b, control = model.state(), model.state(), model.control()
    n, dt = g['n_object'], g['dt']
    q_ref, v_ref = g['q'].copy(), g['v'].copy()
    native_history, reference_history = [g['q'].copy()], [g['q'].copy()]
    for frame in range(1, len(controls)):
        for substep in range(substeps):
            ctrl_q, ctrl_v = sample_controller(controls[frame-1], controls[frame], substep, substeps)
            q, v = a.particle_q.numpy().copy(), a.particle_qd.numpy().copy()
            q[n:], v[n:] = ctrl_q, ctrl_v
            # Do not let a collapsing edge turn into 0/0 inside Newton.
            if np.any(np.linalg.norm(q[g['edges'][:, 1]] - q[g['edges'][:, 0]], axis=1) <= 1e-6):
                raise FloatingPointError('Вырожденная пружина во время rollout')
            a.particle_q.assign(q); a.particle_qd.assign(v)
            native_step(solver, a, b, control, dt)
            q_next, v_next = apply_drag(q, b.particle_qd.numpy(), n, g['drag'], dt)
            if not np.isfinite(q_next).all() or not np.isfinite(v_next).all():
                raise FloatingPointError('Симуляция разошлась')
            b.particle_q.assign(q_next); b.particle_qd.assign(v_next)
            a, b = b, a

            q_ref[n:], v_ref[n:] = ctrl_q, ctrl_v
            f_ref = phystwin_forces_numpy(g, q_ref, v_ref)
            v_ref[:n] += dt * (f_ref[:n] / g['mass'][:n, None] + g['gravity'])
            v_ref[:n] *= np.exp(-dt * g['drag'])
            q_ref[:n] += dt * v_ref[:n]
        native_history.append(a.particle_q.numpy().copy())  # CPU numpy() can share the Warp buffer
        reference_history.append(q_ref.copy())
    return np.stack(native_history), np.stack(reference_history)

def test_rollout():
    global demo_graph, demo_native, demo_reference
    demo_graph = fixture(); demo_graph['drag'] = 3.
    controls = np.repeat(demo_graph['q'][2:][None], 21, axis=0)
    controls[:, :, 2] += np.linspace(0., .02, 21)[:, None]
    demo_native, demo_reference = rollout(demo_graph, controls)
    error = np.linalg.norm(demo_native[:, :2] - demo_reference[:, :2], axis=-1)
    print('Max position error [m]:', error.max())
    assert error.max() < 2e-5
if ready('step', 'controllers', 'drag'):
    check('rollout', test_rollout)

Max position error [m]: 2.926422870917457e-07
PASS rollout


In [20]:
if ready('rollout'):
    def show_frame(frame=0):
        fig = plt.figure(figsize=(10, 4))
        ax = fig.add_subplot(121, projection='3d')
        q, ref = demo_native[frame], demo_reference[frame]
        for i, j in demo_graph['edges']:
            ax.plot(*q[[i, j]].T, color='tab:blue')
        ax.scatter(*q[:2].T, label='Newton', color='tab:blue')
        ax.scatter(*ref[:2].T, label='NumPy reference', marker='x', color='tab:orange')
        ax.scatter(*q[2:].T, label='Controller', color='red')
        ax.set(xlabel='x [m]', ylabel='y [m]', zlabel='z [m]',
               xlim=(-.02, .26), ylim=(-.04, .04), zlim=(.75, .85))
        ax.legend()
        error = np.linalg.norm(demo_native[:, :2] - demo_reference[:, :2], axis=-1).max(axis=1)
        other = fig.add_subplot(122)
        other.plot(np.arange(len(error)) * 20 * demo_graph['dt'], error)
        other.set(xlabel='simulation time [s]', ylabel='max position error [m]')
        other.axvline(frame * 20 * demo_graph['dt'], color='red', alpha=.4)
        plt.tight_layout(); plt.show(); plt.close(fig)
    display(widgets.interactive(show_frame, frame=widgets.IntSlider(min=0, max=len(demo_native)-1, value=0)))
else:
    print('Визуализация появится после завершения rollout.')

interactive(children=(IntSlider(value=0, description='frame', max=20), Output()), _dom_classes=('widget-intera…

## 7. Контакт с землёй — отдельная модель

У PhysTwin плоскость $z=0$ и импульсная коррекция скорости с временем удара.
Штатный контакт частиц SemiImplicit использует силы: перенос `friction` и
`restitution` по совпадению имён не воспроизводит этот алгоритм.

### Задание 6 — один контакт PhysTwin

Реализуй случай Z-up (`reverse_factor=1`), без self-collision. Входная скорость
уже включает силы, гравитацию и drag. При `q.z + v.z*dt < 0` и `v.z < -1e-4`:

- $t_{hit}=-q_z/v_z$;
- $v'_z=-e\,v_z$, где $e$ ограничено диапазоном $[0,1]$;
- $v'_{xy}=a\,v_{xy}$, где
  $a=\max(0,1-\mu(1+e)|v_z|/\max(\|v_{xy}\|,10^{-6}))$, $\mu\in[0,2]$;
- $q'=q+v\,t_{hit}+v'(dt-t_{hit})$.

Иначе просто `q' = q + v*dt`. Исходная точка должна быть над плоскостью.
Для reverse-Z сначала выведи формулы отдельно. Не добавляй одновременно
штатный ground Newton и эту коррекцию: это двойной контакт.

In [21]:
def ground_step(q, v, dt, restitution, friction):
    q, v = np.asarray(q, dtype=float), np.asarray(v, dtype=float)
    if q[2] < 0 or dt <= 0:
        raise ValueError('Начальная точка должна быть над землёй, dt > 0')
    if q[2] + v[2] * dt < 0 and v[2] < -1e-4:
        e, mu = np.clip(restitution, 0., 1.), np.clip(friction, 0., 2.)
        toi = -q[2] / v[2]
        factor = max(0., 1. - mu * (1. + e) * abs(v[2]) / max(np.linalg.norm(v[:2]), 1e-6))
        v_new = np.array([factor * v[0], factor * v[1], -e * v[2]])
        return q + v * toi + v_new * (dt - toi), v_new
    return q + v * dt, v.copy()

<details><summary>Подсказка: раскрыть рабочий вариант после своей попытки</summary>

Скопируй код в ячейку задания и выполни её повторно.

```python
def ground_step(q, v, dt, restitution, friction):
    q, v = np.asarray(q, dtype=float), np.asarray(v, dtype=float)
    if q[2] < 0 or dt <= 0:
        raise ValueError('Начальная точка должна быть над землёй, dt > 0')
    if q[2] + v[2] * dt < 0 and v[2] < -1e-4:
        e, mu = np.clip(restitution, 0., 1.), np.clip(friction, 0., 2.)
        toi = -q[2] / v[2]
        factor = max(0., 1. - mu * (1. + e) * abs(v[2]) / max(np.linalg.norm(v[:2]), 1e-6))
        v_new = np.array([factor * v[0], factor * v[1], -e * v[2]])
        return q + v * toi + v_new * (dt - toi), v_new
    return q + v * dt, v.copy()
```

</details>

In [22]:
def test_ground():
    q, v = ground_step(np.array([0., 0., .01]), np.array([1., 0., -2.]), .01, .5, .2)
    np.testing.assert_allclose(v, [.4, 0., 1.])
    np.testing.assert_allclose(q, [.007, 0., .005])
    q, v = ground_step(np.array([0., 0., .1]), np.array([1., 0., -2.]), .01, .5, .2)
    np.testing.assert_allclose(q, [.01, 0., .08])
    np.testing.assert_allclose(v, [1., 0., -2.])
    q, v = ground_step(np.array([0., 0., .01]), np.array([0., 0., -2.]), .01, 2., 4.)
    np.testing.assert_allclose(q, [0., 0., .01])
    np.testing.assert_allclose(v, [0., 0., 2.])
check('ground', test_ground)

PASS ground


**Как встроить контакт:** после `apply_drag` возьми старую позицию `q[i]` и
новую заторможенную скорость, вызови `ground_step` для каждой динамической частицы
и запиши обе выходные величины. Позицию, возвращённую `apply_drag`, в этом случае
замени результатом контакта. Контроллеры пропусти.

Self-collision оставлена следующим заданием по исходному `object_collision`:
в нём есть маски, порог расстояния, условие сближения, импульсы и усреднение
по числу контактов. Сначала сделай тест двух частиц с равными массами,
потом с разными, затем нескольких одновременных контактов. Включение
`particle_grid` Newton само по себе не переносит этот алгоритм.

## 8. Проверка градиента до оптимизации параметров

Для воспроизведения градиентного этапа PhysTwin нужен не только правильный forward.
Проверим `d(loss)/d(ke)` по центральным конечным разностям на одной пружине.
В этом тесте нет управления, drag и контактов; есть отдельные состояния для
каждого шага. Перезапись одного буфера на протяжении `wp.Tape()` ломает backward.

Сначала готовый kernel скалярной ошибки. Сохраняем его вне репозитория:
так Warp может прочитать исходник при компиляции независимо от frontend notebook.

In [23]:
import importlib.util
import sys

kernel_source = """import warp as wp
@wp.kernel
def position_loss(q: wp.array(dtype=wp.vec3), target: float, loss: wp.array(dtype=float)):
    error = q[0][0] - target
    loss[0] = error * error
"""
module_name = 'lesson_loss_' + hashlib.sha256(kernel_source.encode()).hexdigest()[:12]
module_path = CACHE / (module_name + '.py')
module_path.write_text(kernel_source)
spec = importlib.util.spec_from_file_location(module_name, module_path)
loss_module = importlib.util.module_from_spec(spec)
sys.modules[module_name] = loss_module
spec.loader.exec_module(loss_module)

def spring_loss(stiffness, backward=False):
    g = dict(q=np.array([[.13, 0., .8], [0., 0., .8]]), v=np.zeros((2, 3)),
             mass=np.array([.1, 0.]), n_object=1, edges=np.array([[0, 1]], dtype=np.int32),
             rest=np.array([.1]), log_y=np.log([stiffness * .1]), damping=0.,
             y_min=0., y_max=1e5, drag=0., gravity=np.zeros(3), dt=.001)
    model = build_model(g, requires_grad=backward)
    solver = SolverSemiImplicit(model, enable_tri_contact=False)
    states = [model.state(requires_grad=backward) for _ in range(31)]
    control = model.control()
    loss = wp.zeros(1, dtype=float, device=DEVICE, requires_grad=backward)
    tape = wp.Tape()
    with tape:
        for a, b in zip(states[:-1], states[1:]):
            native_step(solver, a, b, control, g['dt'])
        wp.launch(loss_module.position_loss, dim=1,
                  inputs=[states[-1].particle_q, .11], outputs=[loss], device=DEVICE)
    if backward:
        tape.backward(loss)
        return float(loss.numpy()[0]), float(model.spring_stiffness.grad.numpy()[0])
    return float(loss.numpy()[0])

### Задание 7 — конечная разность

Вычисли центральную разность `(f(k+eps) - f(k-eps)) / (2*eps)`.
Не используй слишком маленькое `eps`: состояние Newton здесь в float32.
После проверки повтори с несколькими `eps` и объясни расхождение.

In [24]:
def finite_difference(function, value, epsilon):
    if epsilon <= 0:
        raise ValueError('epsilon должен быть положительным')
    return (function(value + epsilon) - function(value - epsilon)) / (2 * epsilon)

<details><summary>Подсказка: раскрыть рабочий вариант после своей попытки</summary>

Скопируй код в ячейку задания и выполни её повторно.

```python
def finite_difference(function, value, epsilon):
    if epsilon <= 0:
        raise ValueError('epsilon должен быть положительным')
    return (function(value + epsilon) - function(value - epsilon)) / (2 * epsilon)
```

</details>

In [25]:
def test_gradient():
    # Also tests the exercise independently of Warp.
    np.testing.assert_allclose(finite_difference(lambda x: x*x, 3., .1), 6.)
    loss, autodiff = spring_loss(30., backward=True)
    finite = finite_difference(spring_loss, 30., .1)
    print(dict(loss=loss, autodiff=autodiff, finite_difference=finite))
    assert abs(finite) > 1e-8, 'Тест должен иметь ненулевой градиент'
    np.testing.assert_allclose(autodiff, finite, rtol=.03, atol=1e-8)
if ready('step'):
    check('gradient', test_gradient)

Module lesson_loss_04339a3a7b00 9a120dc load on device 'cuda:0' took 47.48 ms  (compiled)
{'loss': 0.0002531655482016504, 'autodiff': -4.23607252741931e-06, 'finite_difference': -4.239263944327831e-06}
PASS gradient


Градиент по `ke` ещё не равен градиенту по параметру PhysTwin `theta=log(Y)`.
Внутри активной области без clamp:
$\partial L/\partial\theta=(\partial L/\partial k)\,k$.
Для автоматического применения этого правила вычисляй `ke` из `theta` в Warp
**внутри tape**. NumPy-преобразование перед `finalize()` не создаёт такую связь.
На пороге активности и clamp функция негладкая — проверяй их отдельно.

## 9. Переход к твоему объекту PhysTwin

### Задание 8 — экспорт восстановленного графа

Меш не содержит обученных пружин PhysTwin. Понадобятся:

| Данные | Откуда брать после восстановления PhysTwin |
|---|---|
| Объект: начальные позиции и скорости | `simulator.wp_init_vertices`, `wp_init_velocities` |
| Рёбра, длины покоя, массы | `wp_springs`, `wp_rest_lengths`, `wp_masses` |
| Обученные log(Y) | `wp_spring_Y` **после загрузки checkpoint** |
| Управляющие траектории | `simulator.controller_points` |
| dt, substeps, drag, dashpot, границы Y | атрибуты восстановленного `simulator` |
| Оси / направление гравитации | `reverse_factor` и метаданные исходных данных |

Не перестраивай соседство по новому mesh sampling: веса checkpoint индексированы
порядком рёбер. Воспроизводи создание trainer по закреплённому `inference_warp.py`
в **собственном окружении PhysTwin**, с его config и `optimal_params.pkl`.
Используй явный путь к checkpoint, а не первый файл из `glob`.

Пример восстановления весов после создания `trainer` (не запускает `trainer.test`,
который дополнительно выполняет rollout и рендер):

```python
checkpoint = torch.load(checkpoint_path, map_location=cfg.device, weights_only=True)
sim = trainer.simulator
assert len(checkpoint['spring_Y']) == sim.n_springs
sim.set_spring_Y(torch.log(checkpoint['spring_Y']).detach().clone())
sim.set_collide(checkpoint['collide_elas'].detach().clone(),
                checkpoint['collide_fric'].detach().clone())
sim.set_collide_object(checkpoint['collide_object_elas'].detach().clone(),
                      checkpoint['collide_object_fric'].detach().clone())
```

Следующую функцию скопируй в тот процесс **после** восстановления весов.
Здесь она только определяется, чужие pickle/checkpoint автоматически не загружаются.
Проверь единицы и укажи реальную систему координат. Экспортируй в каталог вне репозитория.

In [26]:
def export_phystwin_graph(sim, destination, source_root, checkpoint_path, frame_name):
    """Call in the PhysTwin process after restoring a trusted checkpoint; positions must be in metres."""
    from pathlib import Path
    import json, hashlib, subprocess
    import numpy as np

    destination, source_root, checkpoint_path = map(Path, (destination, source_root, checkpoint_path))
    if not frame_name or frame_name == 'TBD':
        raise ValueError('Укажи проверенную систему координат')
    destination.mkdir(parents=True, exist_ok=False)  # never overwrite an earlier export
    def array(value):
        if hasattr(value, 'detach'):
            return value.detach().cpu().numpy()
        return value.numpy() if hasattr(value, 'numpy') else np.asarray(value)
    def sha(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(chunk)
        return digest.hexdigest()
    q, v = array(sim.wp_init_vertices), array(sim.wp_init_velocities)
    controls = (array(sim.controller_points) if sim.controller_points is not None
                else np.empty((1, 0, 3), dtype=np.float32))
    n = len(q)
    arrays = dict(q=np.concatenate([q, controls[0]]), v=np.concatenate([v, np.zeros_like(controls[0])]),
                  mass=np.concatenate([array(sim.wp_masses), np.zeros(controls.shape[1])]),
                  n_object=n, edges=array(sim.wp_springs), rest=array(sim.wp_rest_lengths),
                  log_y=array(sim.wp_spring_Y), damping=sim.dashpot_damping,
                  y_min=sim.spring_Y_min, y_max=sim.spring_Y_max, drag=sim.drag_damping,
                  gravity=np.array([0., 0., -9.8 * sim.reverse_factor]), dt=sim.dt,
                  controls=controls, substeps=sim.num_substeps,
                  restitution=array(sim.wp_collide_elas), friction=array(sim.wp_collide_fric),
                  object_restitution=array(sim.wp_collide_object_elas),
                  object_friction=array(sim.wp_collide_object_fric),
                  object_collision_enabled=sim.object_collision_flag,
                  collision_dist=sim.collision_dist if sim.object_collision_flag else 0.0)
    if sim.object_collision_flag:
        arrays['collision_masks'] = array(sim.wp_masks)
    output = destination / 'graph.npz'
    np.savez_compressed(output, **arrays)
    source_file = source_root / 'qqtt/model/diff_simulator/spring_mass_warp.py'
    provenance = dict(format='phystwin-newton-lesson-v1', length_unit='m', mass_unit='kg',
                      time_unit='s', coordinate_frame=frame_name, graph_sha256=sha(output),
                      checkpoint_path=str(checkpoint_path), checkpoint_sha256=sha(checkpoint_path),
                      source_file_sha256=sha(source_file),
                      source_commit=subprocess.check_output(['git', '-C', str(source_root), 'rev-parse', 'HEAD'], text=True).strip(),
                      source_status=subprocess.check_output(['git', '-C', str(source_root), 'status', '--porcelain'], text=True),
                      contact_mode='exported_only_not_implemented_in_lesson_rollout')
    (destination / 'provenance.json').write_text(json.dumps(provenance, indent=2))
    return output

Путь `REAL_EXPORT` ниже по умолчанию выключен. Укажи каталог экспорта
на хосте, например `Path.home() / "phystwin-export"`.
Загрузка проверяет хеш, единицы и структуру; она **не запускает** объект.
Для научного запуска дополнительно нужны dataset/config hashes и полный manifest
по AGENTS.md: этот минимальный учебный экспорт не заменяет их.

In [ ]:
REAL_EXPORT = None  # например Path('/phystwin-export') с graph.npz и provenance.json

def load_export(folder):
    folder = Path(folder)
    provenance = json.loads((folder / 'provenance.json').read_text())
    if provenance['format'] != 'phystwin-newton-lesson-v1':
        raise ValueError('Неизвестный формат экспорта')
    if (provenance['length_unit'], provenance['mass_unit'], provenance['time_unit']) != ('m', 'kg', 's'):
        raise ValueError('Нужны метры, килограммы и секунды; молчаливой конверсии нет')
    digest = hashlib.sha256()
    with (folder / 'graph.npz').open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    if digest.hexdigest() != provenance['graph_sha256']:
        raise ValueError('Хеш graph.npz не совпадает')
    with np.load(folder / 'graph.npz', allow_pickle=False) as archive:
        g = {name: archive[name].copy() for name in archive.files}
    for name in ('n_object', 'substeps', 'object_collision_enabled'):
        g[name] = int(g[name].item())
    for name in ('dt', 'drag', 'damping', 'y_min', 'y_max'):
        g[name] = float(g[name].item())
    validate_graph(g)
    if g['substeps'] <= 0 or g['controls'].ndim != 3 or g['controls'].shape[1:] != (len(g['q']) - g['n_object'], 3):
        raise ValueError('Неверная форма controls или число substeps')
    if len(g['controls']) == 0 or not np.isfinite(g['controls']).all():
        raise ValueError('Пустые или нефинитные controls')
    np.testing.assert_allclose(g['q'][g['n_object']:], g['controls'][0])
    return g, provenance

if REAL_EXPORT is None:
    print('Реальные данные не выбраны. Учебные проверки используют только синтетическую сцену.')
else:
    real_graph, provenance = load_export(REAL_EXPORT)
    print({key: provenance[key] for key in ('source_commit', 'graph_sha256', 'coordinate_frame')})
    print('Objects:', real_graph['n_object'], 'Springs:', len(real_graph['edges']))
    print('Original object collisions enabled:', real_graph['object_collision_enabled'])
    print('Экспорт проверен; full-physics parity ещё не проверена.')

## 10. Финальное задание: проверка на исходных Warp kernels

NumPy-проверки выше подтверждают соответствие явно записанным формулам.
Следующий уровень — получить эталон из **самого PhysTwin**:

1. Восстанови один и тот же объект и checkpoint; сохрани хеши данных и графа.
2. Сравни `eval_springs` и силы Newton на одинаковых состояниях **до интегрирования**.
3. Сравни один подшаг без контактов, затем кадр при одинаковых `dt` и `num_substeps`.
   Проверяй `frame_dt = dt * num_substeps`, а не подменяй его автоматически `1/FPS`.
4. Отдельно добавь контроллеры, drag, ground, затем object/self-collision.
   Для каждого сравнения сохрани `q`, `v`, ошибки и параметры. Не подгоняй жёсткость,
   чтобы скрыть ошибку переноса.
5. Перепиши NumPy-коррекции как Warp kernels и проверь их градиенты.
6. Только после проверки адаптера составляй план калибровки по реальным RGB-D.
   Зафиксируй геометрию, раздели целые траектории на calibration и held-out test,
   затем сравни некалиброванный и калиброванный варианты по одному протоколу.

**Критерий готовности адаптера:** понятны и протестированы все несовпадения;
перенос не считается точным только потому, что объект правдоподобно движется.

### Что ещё не реализовано в этом уроке

- Self-collision / object collision PhysTwin и их ускоренная структура соседства.
- Дифференцируемые версии drag и ground, включая поведение на границах контакта.
- Полный RGB-D loss, оптимизатор и обучение Gaussian-визуализации.
- Подключение к Isaac Lab и двустороннее взаимодействие с роботом.

Эти пункты — задания следующего этапа, а не заявленные возможности адаптера.
После урока полезно вынести свои функции в `src/lab/adapters/`, а проверки — в `tests/`.

In [ ]:
required = {'coefficients', 'builder', 'step', 'controllers', 'drag', 'rollout', 'ground', 'gradient'}
print('Завершено:', len(passed & required), '/', len(required))
print('Осталось:', ', '.join(sorted(required - passed)) or 'учебные проверки завершены')
print('Экспорт реального PhysTwin, исходные kernels, контакты нескольких объектов и RGB-D калибровка проверяются отдельно.')